# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SUKRIT004/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Baseline rule: prioritize pages that have meaningful search exposure and a CTR below the expected CTR for their position tier. The score will increase when the CTR gap is larger and when the page has more impressions, because a larger gap with more exposure represents a potentially more useful review opportunity. The rule is intended to rank pages for human review, not to predict whether a change will definitely improve performance.

Reason code: LOW_CTR_VS_POSITION

Action label: REVIEW_CTR

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/SUKRIT004/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

# Only pages with search exposure and a valid position
visible = df[
    (df["impressions_90d"] > 0) &
    (df["avg_position"] > 0)
].copy()

print("Visible rows:", len(visible))

# ---------------------------------------------------------
# Signal 1: CTR relative to search position
# FlyRank flag-linked signal: CTR-vs-position logic
# ---------------------------------------------------------

position_check = (
    visible.groupby("position_tier")
    .agg(
        mean_ctr=("ctr", "mean"),
        median_ctr=("ctr", "median"),
        n=("ctr", "size")
    )
    .sort_values("mean_ctr", ascending=False)
)

print("\n=== Signal 1: CTR vs position ===")
display(position_check)

# ---------------------------------------------------------
# Signal 2: Search volume / impressions
# FlyRank quick-win / volume signal
# ---------------------------------------------------------

visible["impression_bucket"] = pd.qcut(
    visible["impressions_90d"],
    q=3,
    labels=["low", "medium", "high"],
    duplicates="drop"
)

volume_check = (
    visible.groupby("impression_bucket", observed=True)
    .agg(
        median_impressions=("impressions_90d", "median"),
        mean_ctr=("ctr", "mean"),
        n=("ctr", "size")
    )
    .reset_index()
)

print("\n=== Signal 2: Search volume ===")
display(volume_check)

Visible rows: 28795

=== Signal 1: CTR vs position ===


,mean_ctr,median_ctr,n
position_tier,,,
top_3,2.764453,0.00,1116
page_1,0.652467,0.16,11814
striking,0.323239,0.11,7304
page_3_5,0.222484,0.03,7242
deep,0.150212,0.00,1319



=== Signal 2: Search volume ===


,impression_bucket,median_impressions,mean_ctr,n
0,low,39.0,1.050705,9603
1,medium,829.0,0.205650,9593
2,high,7023.0,0.302214,9599


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2 — Build the transparent baseline queue

baseline = visible.copy()

# Expected CTR = mean CTR for the page's position tier
expected_ctr = (
    baseline.groupby("position_tier")["ctr"]
    .mean()
    .rename("expected_ctr")
)

baseline = baseline.join(expected_ctr, on="position_tier")

# Positive gap = observed CTR is below the typical CTR for its tier
baseline["ctr_gap"] = (
    baseline["expected_ctr"] - baseline["ctr"]
)

# Give larger-volume pages more weight without letting volume dominate
baseline["opportunity_score"] = (
    baseline["ctr_gap"].clip(lower=0)
    * np.log1p(baseline["impressions_90d"])
)

# One reason code and one action label
baseline["reason_code"] = np.where(
    baseline["opportunity_score"] > 0,
    "LOW_CTR_VS_POSITION",
    "NONE"
)

baseline["action"] = np.where(
    baseline["opportunity_score"] > 0,
    "REVIEW_CTR",
    "NO_ACTION"
)

# Rank highest opportunities first
baseline = baseline.sort_values(
    "opportunity_score",
    ascending=False
).reset_index(drop=True)

baseline["rank"] = np.arange(1, len(baseline) + 1)

# Required output columns
output_cols = [
    "rank",
    "content_type",
    "impressions_90d",
    "ctr",
    "avg_position",
    "position_tier",
    "expected_ctr",
    "ctr_gap",
    "opportunity_score",
    "reason_code",
    "action"
]

queue = baseline[output_cols].copy()

display(queue.head(20))

# Write the required CSV
from pathlib import Path

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

queue.to_csv(output_path, index=False)

print(f"\nWrote baseline queue to: {output_path}")
print(f"Rows in queue: {len(queue):,}")

,rank,content_type,impressions_90d,ctr,avg_position,position_tier,expected_ctr,ctr_gap,opportunity_score,reason_code,action
0,1,keyword article,509252,0.15,2.5,top_3,2.764453,2.614453,34.355748,LOW_CTR_VS_POSITION,REVIEW_CTR
1,2,keyword article,272144,0.03,2.3,top_3,2.764453,2.734453,34.219197,LOW_CTR_VS_POSITION,REVIEW_CTR
2,3,keyword article,128068,0.01,2.2,top_3,2.764453,2.754453,32.393266,LOW_CTR_VS_POSITION,REVIEW_CTR
3,4,keyword article,149712,0.07,2.9,top_3,2.764453,2.694453,32.108388,LOW_CTR_VS_POSITION,REVIEW_CTR
4,5,keyword article,463103,0.41,2.3,top_3,2.764453,2.354453,30.715509,LOW_CTR_VS_POSITION,REVIEW_CTR
5,6,keyword article,52687,0.08,2.6,top_3,2.764453,2.684453,29.185761,LOW_CTR_VS_POSITION,REVIEW_CTR
6,7,keyword article,73675,0.19,2.9,top_3,2.764453,2.574453,28.853012,LOW_CTR_VS_POSITION,REVIEW_CTR
7,8,keyword article,43650,0.14,0.7,top_3,2.764453,2.624453,28.039612,LOW_CTR_VS_POSITION,REVIEW_CTR
8,9,keyword article,29747,0.07,1.2,top_3,2.764453,2.694453,27.754264,LOW_CTR_VS_POSITION,REVIEW_CTR
9,10,keyword article,26470,0.05,0.7,top_3,2.764453,2.714453,27.643464,LOW_CTR_VS_POSITION,REVIEW_CTR



Wrote baseline queue to: work/outputs/baseline_action_score.csv
Rows in queue: 28,795


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top-20 review shows that the baseline consistently prioritizes pages with CTR below the observed average for their position tier. The recommendations are useful as a review queue, but they should not be interpreted as guaranteed optimization opportunities. A page could have a low CTR for legitimate reasons, and position-tier averages may not account for query intent, content differences, or other factors.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 — Top-20 review table

# Section 3 — Top-20 skeptical review

top20 = queue.head(20).copy()

reviews = []

for _, row in top20.iterrows():
    if row["impressions_90d"] >= 100000:
        confidence = "High exposure makes this a useful review candidate."
    elif row["impressions_90d"] >= 30000:
        confidence = "Moderate exposure makes the opportunity worth checking."
    else:
        confidence = "Lower exposure means the opportunity should be reviewed cautiously."

    wrong = (
        f"Could be wrong if the position-tier average is not appropriate "
        f"for this page's query or content context."
    )

    reviews.append({
        "rank": int(row["rank"]),
        "action": row["action"],
        "reason_code": row["reason_code"],
        "confidence_note": confidence,
        "what_would_make_it_wrong": wrong
    })

review_df = pd.DataFrame(reviews)

display(review_df)

,rank,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,REVIEW_CTR,LOW_CTR_VS_POSITION,High exposure makes this a useful review candi...,Could be wrong if the position-tier average is...
1,2,REVIEW_CTR,LOW_CTR_VS_POSITION,High exposure makes this a useful review candi...,Could be wrong if the position-tier average is...
2,3,REVIEW_CTR,LOW_CTR_VS_POSITION,High exposure makes this a useful review candi...,Could be wrong if the position-tier average is...
3,4,REVIEW_CTR,LOW_CTR_VS_POSITION,High exposure makes this a useful review candi...,Could be wrong if the position-tier average is...
4,5,REVIEW_CTR,LOW_CTR_VS_POSITION,High exposure makes this a useful review candi...,Could be wrong if the position-tier average is...
5,6,REVIEW_CTR,LOW_CTR_VS_POSITION,Moderate exposure makes the opportunity worth ...,Could be wrong if the position-tier average is...
6,7,REVIEW_CTR,LOW_CTR_VS_POSITION,Moderate exposure makes the opportunity worth ...,Could be wrong if the position-tier average is...
7,8,REVIEW_CTR,LOW_CTR_VS_POSITION,Moderate exposure makes the opportunity worth ...,Could be wrong if the position-tier average is...
8,9,REVIEW_CTR,LOW_CTR_VS_POSITION,Lower exposure means the opportunity should be...,Could be wrong if the position-tier average is...
9,10,REVIEW_CTR,LOW_CTR_VS_POSITION,Lower exposure means the opportunity should be...,Could be wrong if the position-tier average is...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some weak picks are likely to occur when the position-tier average is not a good expectation for an individual page. In particular, the rule can prioritize high-impression pages with low CTR even when the low CTR may be appropriate for the page's content or query mix. The rule does not use future-window outcomes, trend_direction, or any label-derived field. The score is based only on current ctr, avg_position, position_tier, and impressions_90d.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — leakage check

forbidden = [
    "trend_direction",
    "declining",
    "label",
    "future_ctr",
    "future_position"
]

present_forbidden = [
    col for col in forbidden
    if col in queue.columns
]

print("Forbidden/future columns found in queue:", present_forbidden)

assert not present_forbidden, (
    f"Potential leakage detected: {present_forbidden}"
)

print("Leakage check passed: no listed future/label-derived columns are in the queue.")

Forbidden/future columns found in queue: []
Leakage check passed: no listed future/label-derived columns are in the queue.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.